# Comprehensive Climate Data Integrity Checks
## Full validation including Temperature Max/Min checks
### Enhanced with Percentage Calculations and Visualizations

In [ ]:
# Import required libraries
import json
import warnings
from datetime import datetime

import boto3
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

warnings.filterwarnings("ignore")

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

print("Libraries imported successfully")

## 1. Load ALL Variables from ERA5 Dataset

In [ ]:
print("=" * 60)
print("LOADING ERA5 DATASET WITH ALL VARIABLES")
print("=" * 60)

# Load ERA5 dataset with all variables
storage = icechunk.s3_storage(
    bucket="carbonplan-srm",
    prefix="input/tensor/era5_rechunked_resampled.icechunk",
    from_env=True,
)
repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")
era5_full = xr.open_zarr(session.store, consolidated=False)

print("\nAvailable ERA5 variables:")
for var in era5_full.data_vars:
    print(f"  - {var}")

# Extract individual variables
ERA5_vars = {}

# Subset to South Africa region and time period
time_slice = slice("1978-01-01", "2014-12-31")
lat_slice = slice(-22, -35)
lon_slice = slice(16, 33)

print("\nExtracting and processing ERA5 variables...")

# Precipitation
ERA5_vars["precipitation"] = (
    era5_full["mean_total_precipitation_rate"].sel(
        time=time_slice, latitude=lat_slice, longitude=lon_slice
    )
    * 86400.0
)  # Convert to mm/day
ERA5_vars["precipitation"].attrs["units"] = "mm/day"
print(f"  ✓ Precipitation loaded: shape {ERA5_vars['precipitation'].shape}")

# Temperature max
ERA5_vars["tmax"] = era5_full[
    "maximum_2m_temperature_since_previous_post_processing"
].sel(time=time_slice, latitude=lat_slice, longitude=lon_slice)
ERA5_vars["tmax"].attrs["units"] = "K"  # Kelvin
print(f"  ✓ Temperature max loaded: shape {ERA5_vars['tmax'].shape}")

# Temperature min
ERA5_vars["tmin"] = era5_full[
    "minimum_2m_temperature_since_previous_post_processing"
].sel(time=time_slice, latitude=lat_slice, longitude=lon_slice)
ERA5_vars["tmin"].attrs["units"] = "K"  # Kelvin
print(f"  ✓ Temperature min loaded: shape {ERA5_vars['tmin'].shape}")

# Average temperature
ERA5_vars["temperature"] = era5_full["2m_temperature"].sel(
    time=time_slice, latitude=lat_slice, longitude=lon_slice
)
ERA5_vars["temperature"].attrs["units"] = "K"  # Kelvin
print(f"  ✓ Temperature (2m) loaded: shape {ERA5_vars['temperature'].shape}")

print(
    f"\nERA5 time range: {ERA5_vars['precipitation'].time.values[0]} to {ERA5_vars['precipitation'].time.values[-1]}"
)

## 2. Load CESM2-WACCM Historical Dataset

In [ ]:
print("=" * 60)
print("LOADING CESM2-WACCM HISTORICAL DATASET")
print("=" * 60)

# Load HIST dataset
storage = icechunk.s3_storage(
    bucket="carbonplan-srm",
    prefix="input/tensor/CESM2-WACCM-Historical/icechunk/icechunk",
    from_env=True,
)

repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")
hist_full = xr.open_zarr(session.store, consolidated=False)

print("\nAvailable CESM2-WACCM variables (first 20):")
all_vars = list(hist_full.data_vars)
for var in all_vars[:20]:
    print(f"  - {var}")
if len(all_vars) > 20:
    print(f"  ... and {len(all_vars) - 20} more variables")

# Look for specific climate variables
HIST_vars = {}

# Process time dimension first
if "PRECT" in hist_full.data_vars:
    temp_var = hist_full["PRECT"]
    times = pd.to_datetime(temp_var.time.values.astype(str))
    unique_times, index = np.unique(times, return_index=True)

print("\nSearching for and loading CESM2-WACCM variables...")

# Precipitation
if "PRECT" in hist_full.data_vars:
    HIST_vars["precipitation"] = hist_full["PRECT"].isel(time=np.sort(index))
    HIST_vars["precipitation"]["time"] = unique_times
    HIST_vars["precipitation"] = (
        HIST_vars["precipitation"].sel(
            time=slice("1978-01-01", "2014-12-31"),
            lat=slice(-35, -22),
            lon=slice(16, 33),
        )
        * 1000.0
        * 86400.0
    )  # Convert m/s to mm/day
    HIST_vars["precipitation"].attrs["units"] = "mm/day"
    print(f"  ✓ Precipitation (PRECT) loaded: shape {HIST_vars['precipitation'].shape}")

# Look for temperature variables
temp_vars_to_check = [
    ("TREFHTMX", "tmax"),  # Temperature max at reference height
    ("TREFHTMN", "tmin"),  # Temperature min at reference height
    ("TREFHT", "temperature"),  # Temperature at reference height
    ("TS", "surface_temp"),  # Surface temperature
    ("RHREFHT", "relative_humidity"),  # Relative humidity at reference height
]

for var_name, var_key in temp_vars_to_check:
    if var_name in hist_full.data_vars:
        HIST_vars[var_key] = hist_full[var_name].isel(time=np.sort(index))
        HIST_vars[var_key]["time"] = unique_times
        HIST_vars[var_key] = HIST_vars[var_key].sel(
            time=slice("1978-01-01", "2014-12-31"),
            lat=slice(-35, -22),
            lon=slice(16, 33),
        )
        print(f"  ✓ {var_key} ({var_name}) loaded: shape {HIST_vars[var_key].shape}")
    else:
        print(f"  ✗ {var_key} ({var_name}) not found")

if HIST_vars:
    first_var = list(HIST_vars.values())[0]
    print(
        f"\nHIST time range: {first_var.time.values[0]} to {first_var.time.values[-1]}"
    )

## 3. Load BCSD Downscaled Dataset

In [ ]:
print("=" * 60)
print("LOADING BCSD DOWNSCALED DATASET")
print("=" * 60)


s3 = boto3.client("s3")

bucket = "carbonplan-srm"
BCSD_vars = {}

# Check for different BCSD files
bcsd_files = [
    "output/BCSD/cesm2-waccm-historical/precipitation.zarr",
    "output/BCSD/cesm2-waccm-historical/tasmax.zarr",
    "output/BCSD/cesm2-waccm-historical/tasmin.zarr",
    "output/BCSD/cesm2-waccm-historical/tas.zarr",
]

var_mapping = {
    "precipitation": "precipitation",
    "tasmax": "tmax",
    "tasmin": "tmin",
    "tas": "temperature",
}

print("\nSearching for BCSD variables...")
for file_path in bcsd_files:
    var_name = file_path.split("/")[-1].replace(".zarr", "")

    try:
        # List objects to check if zarr exists
        response = s3.list_objects_v2(Bucket=bucket, Prefix=file_path, MaxKeys=1)

        if "Contents" in response:
            # Load the variable
            ds = xr.open_zarr(
                f"s3://{bucket}/{file_path}", storage_options={"anon": False}
            )

            # Get the data variable (usually the first one)
            data_var = list(ds.data_vars)[0]
            data = ds[data_var]

            # Subset to region and time
            data = data.sel(
                time=slice("1978-01-01", "2014-12-31"),
                lat=slice(-35, -22),
                lon=slice(16, 33),
            )

            # Store with standardized name
            standard_name = var_mapping.get(var_name, var_name)
            BCSD_vars[standard_name] = data

            print(f"  ✓ {standard_name} loaded: shape {data.shape}")
        else:
            print(f"  ✗ {var_name} not found")
    except Exception as e:
        print(f"  ✗ {var_name} error: {str(e)[:50]}")

if BCSD_vars:
    first_var = list(BCSD_vars.values())[0]
    print(
        f"\nBCSD time range: {first_var.time.values[0]} to {first_var.time.values[-1]}"
    )

## 4. Data Quality Checks with Detailed Statistics

In [ ]:
print("=" * 60)
print("DATA QUALITY CHECKS WITH PERCENTAGES")
print("=" * 60)

# Combine all datasets
all_datasets = {"ERA5": ERA5_vars, "HIST": HIST_vars, "BCSD": BCSD_vars}

# Dictionary to store all statistics
quality_stats = {}

for dataset_name, variables in all_datasets.items():
    if not variables:
        continue

    print(f"\n{dataset_name} DATASET")
    print("-" * 40)

    quality_stats[dataset_name] = {}

    for var_name, var_data in variables.items():
        print(f"\n  {var_name.upper()}:")

        # Get array values
        values = var_data.values
        total_points = values.size

        # Initialize stats dictionary
        stats = {"total_points": total_points, "shape": var_data.shape}

        # Missing values
        missing = np.isnan(values).sum()
        missing_pct = (missing / total_points) * 100
        stats["missing_count"] = int(missing)
        stats["missing_pct"] = float(missing_pct)
        print(f"    Missing values: {missing:,} ({missing_pct:.2f}%)")

        # Valid values (for subsequent calculations)
        valid_values = values[~np.isnan(values)]
        valid_count = len(valid_values)
        stats["valid_count"] = int(valid_count)

        if valid_count > 0:
            # Basic statistics
            stats["mean"] = float(np.mean(valid_values))
            stats["min"] = float(np.min(valid_values))
            stats["max"] = float(np.max(valid_values))
            stats["std"] = float(np.std(valid_values))

            # Precipitation-specific checks
            if var_name == "precipitation":
                # Negative values
                negative = (valid_values < 0).sum()
                negative_pct = (negative / valid_count) * 100
                stats["negative_count"] = int(negative)
                stats["negative_pct"] = float(negative_pct)
                print(f"    Negative values: {negative:,} ({negative_pct:.4f}%)")

                # Zero values
                zeros = (valid_values == 0).sum()
                zeros_pct = (zeros / valid_count) * 100
                stats["zero_count"] = int(zeros)
                stats["zero_pct"] = float(zeros_pct)
                print(f"    Zero values: {zeros:,} ({zeros_pct:.2f}%)")

                # Extreme values (>100 mm/day)
                extreme = (valid_values > 100).sum()
                extreme_pct = (extreme / valid_count) * 100
                stats["extreme_count"] = int(extreme)
                stats["extreme_pct"] = float(extreme_pct)
                print(
                    f"    Extreme values (>100 mm/day): {extreme:,} ({extreme_pct:.3f}%)"
                )

                # Very extreme values (>200 mm/day)
                very_extreme = (valid_values > 200).sum()
                very_extreme_pct = (very_extreme / valid_count) * 100
                stats["very_extreme_count"] = int(very_extreme)
                stats["very_extreme_pct"] = float(very_extreme_pct)
                print(
                    f"    Very extreme values (>200 mm/day): {very_extreme:,} ({very_extreme_pct:.4f}%)"
                )

            # Temperature-specific checks
            if var_name in ["tmax", "tmin", "temperature"]:
                # Convert to Celsius if in Kelvin
                if np.mean(valid_values) > 200:
                    temp_c = valid_values - 273.15
                    unit = "°C (converted from K)"
                else:
                    temp_c = valid_values
                    unit = "°C"

                # Unrealistic cold temperatures (<-50°C)
                very_cold = (temp_c < -50).sum()
                very_cold_pct = (very_cold / valid_count) * 100
                stats["very_cold_count"] = int(very_cold)
                stats["very_cold_pct"] = float(very_cold_pct)
                print(
                    f"    Unrealistic cold (<-50°C): {very_cold:,} ({very_cold_pct:.4f}%)"
                )

                # Unrealistic hot temperatures (>55°C)
                very_hot = (temp_c > 55).sum()
                very_hot_pct = (very_hot / valid_count) * 100
                stats["very_hot_count"] = int(very_hot)
                stats["very_hot_pct"] = float(very_hot_pct)
                print(
                    f"    Unrealistic hot (>55°C): {very_hot:,} ({very_hot_pct:.4f}%)"
                )

        # Store stats
        quality_stats[dataset_name][var_name] = stats

# Check Tmax >= Tmin consistency
print("\n" + "=" * 60)
print("TEMPERATURE CONSISTENCY CHECKS")
print("=" * 60)

for dataset_name, variables in all_datasets.items():
    if "tmax" in variables and "tmin" in variables:
        print(f"\n{dataset_name}:")
        tmax = variables["tmax"].values
        tmin = variables["tmin"].values

        # Count violations
        violations = (tmax < tmin).sum()
        valid_comparisons = (~np.isnan(tmax) & ~np.isnan(tmin)).sum()
        violation_pct = (
            (violations / valid_comparisons) * 100 if valid_comparisons > 0 else 0
        )

        print(f"  Tmax < Tmin violations: {violations:,} ({violation_pct:.4f}%)")
        print(f"  Valid comparisons: {valid_comparisons:,}")

        # Store in stats
        if dataset_name not in quality_stats:
            quality_stats[dataset_name] = {}
        quality_stats[dataset_name]["temp_consistency"] = {
            "violations": int(violations),
            "violation_pct": float(violation_pct),
            "valid_comparisons": int(valid_comparisons),
        }

## 5. Visualization of Quality Metrics

In [ ]:
print("=" * 60)
print("GENERATING QUALITY VISUALIZATIONS")
print("=" * 60)

# Create figure for precipitation quality metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Precipitation Quality Metrics Across Datasets", fontsize=16, fontweight="bold"
)

# Prepare data for plotting
datasets_with_precip = []
negative_pcts = []
zero_pcts = []
extreme_pcts = []
missing_pcts = []

for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if dataset_name in quality_stats and "precipitation" in quality_stats[dataset_name]:
        stats = quality_stats[dataset_name]["precipitation"]
        datasets_with_precip.append(dataset_name)
        negative_pcts.append(stats.get("negative_pct", 0))
        zero_pcts.append(stats.get("zero_pct", 0))
        extreme_pcts.append(stats.get("extreme_pct", 0))
        missing_pcts.append(stats.get("missing_pct", 0))

if datasets_with_precip:
    x_pos = np.arange(len(datasets_with_precip))

    # Plot 1: Negative values percentage
    ax1 = axes[0, 0]
    bars1 = ax1.bar(
        x_pos,
        negative_pcts,
        color=["#e74c3c", "#3498db", "#2ecc71"][: len(datasets_with_precip)],
    )
    ax1.set_xlabel("Dataset", fontsize=11)
    ax1.set_ylabel("Percentage (%)", fontsize=11)
    ax1.set_title("Negative Precipitation Values", fontsize=12, fontweight="bold")
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(datasets_with_precip)
    # Add percentage labels on bars
    for i, (bar, pct) in enumerate(zip(bars1, negative_pcts)):
        height = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{pct:.4f}%",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    # Plot 2: Zero values percentage
    ax2 = axes[0, 1]
    bars2 = ax2.bar(
        x_pos,
        zero_pcts,
        color=["#e74c3c", "#3498db", "#2ecc71"][: len(datasets_with_precip)],
    )
    ax2.set_xlabel("Dataset", fontsize=11)
    ax2.set_ylabel("Percentage (%)", fontsize=11)
    ax2.set_title("Zero Precipitation Values", fontsize=12, fontweight="bold")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(datasets_with_precip)
    for i, (bar, pct) in enumerate(zip(bars2, zero_pcts)):
        height = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{pct:.2f}%",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    # Plot 3: Extreme values percentage (>100 mm/day)
    ax3 = axes[1, 0]
    bars3 = ax3.bar(
        x_pos,
        extreme_pcts,
        color=["#e74c3c", "#3498db", "#2ecc71"][: len(datasets_with_precip)],
    )
    ax3.set_xlabel("Dataset", fontsize=11)
    ax3.set_ylabel("Percentage (%)", fontsize=11)
    ax3.set_title("Extreme Precipitation (>100 mm/day)", fontsize=12, fontweight="bold")
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(datasets_with_precip)
    for i, (bar, pct) in enumerate(zip(bars3, extreme_pcts)):
        height = bar.get_height()
        ax3.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{pct:.3f}%",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    # Plot 4: Missing values percentage
    ax4 = axes[1, 1]
    bars4 = ax4.bar(
        x_pos,
        missing_pcts,
        color=["#e74c3c", "#3498db", "#2ecc71"][: len(datasets_with_precip)],
    )
    ax4.set_xlabel("Dataset", fontsize=11)
    ax4.set_ylabel("Percentage (%)", fontsize=11)
    ax4.set_title("Missing Data", fontsize=12, fontweight="bold")
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(datasets_with_precip)
    for i, (bar, pct) in enumerate(zip(bars4, missing_pcts)):
        height = bar.get_height()
        ax4.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{pct:.2f}%",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

plt.tight_layout()
plt.savefig("precipitation_quality_metrics.png", dpi=300, bbox_inches="tight")
print("\n✓ Saved: precipitation_quality_metrics.png")
plt.close()

# Create figure for temperature consistency
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
fig.suptitle(
    "Temperature Consistency: Tmax < Tmin Violations", fontsize=14, fontweight="bold"
)

datasets_with_temp = []
violation_pcts = []

for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if (
        dataset_name in quality_stats
        and "temp_consistency" in quality_stats[dataset_name]
    ):
        datasets_with_temp.append(dataset_name)
        violation_pcts.append(
            quality_stats[dataset_name]["temp_consistency"]["violation_pct"]
        )

if datasets_with_temp:
    x_pos = np.arange(len(datasets_with_temp))
    bars = ax.bar(
        x_pos,
        violation_pcts,
        color=["#e74c3c", "#3498db", "#2ecc71"][: len(datasets_with_temp)],
    )
    ax.set_xlabel("Dataset", fontsize=12)
    ax.set_ylabel("Violation Percentage (%)", fontsize=12)
    ax.set_title("Percentage of points where Tmax < Tmin", fontsize=11)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(datasets_with_temp)

    # Add percentage labels on bars
    for bar, pct in zip(bars, violation_pcts):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{pct:.4f}%",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("temperature_consistency.png", dpi=300, bbox_inches="tight")
print("✓ Saved: temperature_consistency.png")
plt.close()

# Create comprehensive summary table
fig, ax = plt.subplots(figsize=(16, 10))
ax.axis("tight")
ax.axis("off")

# Prepare table data
table_data = []
table_data.append(
    [
        "Dataset",
        "Variable",
        "Total Points",
        "Missing %",
        "Negative %",
        "Zero %",
        "Extreme %\n(>100mm/day)",
        "Very Extreme %\n(>200mm/day)",
    ]
)

for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if dataset_name in quality_stats:
        for var_name, stats in quality_stats[dataset_name].items():
            if var_name == "temp_consistency":
                continue

            row = [dataset_name, var_name, f"{stats['total_points']:,}"]
            row.append(f"{stats.get('missing_pct', 0):.2f}%")

            if var_name == "precipitation":
                row.append(f"{stats.get('negative_pct', 0):.4f}%")
                row.append(f"{stats.get('zero_pct', 0):.2f}%")
                row.append(f"{stats.get('extreme_pct', 0):.3f}%")
                row.append(f"{stats.get('very_extreme_pct', 0):.4f}%")
            else:
                row.extend(["-", "-", "-", "-"])

            table_data.append(row)

# Create table
table = ax.table(
    cellText=table_data,
    cellLoc="center",
    loc="center",
    colWidths=[0.12, 0.15, 0.13, 0.10, 0.11, 0.09, 0.14, 0.16],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.5)

# Style header row
for i in range(len(table_data[0])):
    table[(0, i)].set_facecolor("#3498db")
    table[(0, i)].set_text_props(weight="bold", color="white")

# Color code rows by dataset
colors = {"ERA5": "#ffe6e6", "HIST": "#e6f2ff", "BCSD": "#e6ffe6"}
for i in range(1, len(table_data)):
    dataset = table_data[i][0]
    for j in range(len(table_data[0])):
        table[(i, j)].set_facecolor(colors.get(dataset, "white"))

plt.title(
    "Comprehensive Data Quality Summary Table", fontsize=14, fontweight="bold", pad=20
)
plt.savefig("quality_summary_table.png", dpi=300, bbox_inches="tight")
print("✓ Saved: quality_summary_table.png")
plt.close()

print("\n" + "=" * 60)
print("ALL VISUALIZATIONS SAVED SUCCESSFULLY")
print("=" * 60)
print("\nGenerated files:")
print("  1. precipitation_quality_metrics.png")
print("  2. temperature_consistency.png")
print("  3. quality_summary_table.png")

## 6. Cross-Dataset Comparisons

In [ ]:
print("=" * 60)
print("CROSS-DATASET COMPARISONS")
print("=" * 60)

# Compare common variables across datasets
variable_types = ["precipitation", "tmax", "tmin", "temperature"]

for var_type in variable_types:
    datasets_with_var = [
        (name, vars[var_type])
        for name, vars in all_datasets.items()
        if var_type in vars
    ]

    if len(datasets_with_var) > 1:
        print(f"\n{var_type.upper()} Comparison:")
        print("-" * 40)

        # Calculate statistics for each dataset
        for name, var in datasets_with_var:
            mean_val = float(np.nanmean(var.values))

            # Convert temperature to Celsius for display
            if var_type in ["tmax", "tmin", "temperature"] and mean_val > 200:
                mean_val -= 273.15
                unit = "°C"
            elif var_type == "precipitation":
                unit = "mm/day"
            else:
                unit = ""

            print(f"  {name}: Mean = {mean_val:.2f} {unit}")

        # Calculate differences between datasets
        if len(datasets_with_var) == 2:
            name1, var1 = datasets_with_var[0]
            name2, var2 = datasets_with_var[1]

            # Need to interpolate to common grid for comparison
            try:
                # Use ERA5 grid as reference if available
                if name1 == "ERA5":
                    var2_interp = var2.interp_like(var1, method="linear")
                    diff = float(
                        np.nanmean(var1.values) - np.nanmean(var2_interp.values)
                    )
                elif name2 == "ERA5":
                    var1_interp = var1.interp_like(var2, method="linear")
                    diff = float(
                        np.nanmean(var1_interp.values) - np.nanmean(var2.values)
                    )
                else:
                    # Interpolate both to first dataset's grid
                    var2_interp = var2.interp_like(var1, method="linear")
                    diff = float(
                        np.nanmean(var1.values) - np.nanmean(var2_interp.values)
                    )

                if var_type in ["tmax", "tmin", "temperature"] and abs(diff) > 100:
                    # Likely comparing Kelvin and Celsius
                    diff = diff - 273.15 if diff > 0 else diff + 273.15

                print(f"  Difference ({name1} - {name2}): {diff:.3f} {unit}")
            except Exception as e:
                print(e)
                print("  Could not compute difference due to grid mismatch")

## 7. Final Summary Report

In [ ]:
print("=" * 60)
print("INTEGRITY CHECK SUMMARY REPORT")
print("=" * 60)

# Summary of all checks
summary = {
    "timestamp": datetime.now().isoformat(),
    "datasets_checked": list(all_datasets.keys()),
    "checks_performed": [],
    "issues_found": [],
    "recommendations": [],
    "quality_statistics": quality_stats,
}

print("\n✓ CHECKS PERFORMED:")
print("-" * 40)

# List what was checked
checks = [
    "1. Basic file and variable presence",
    "2. Dimensional consistency",
    "3. Missing data analysis with percentages",
    "4. Precipitation: negative values percentage",
    "5. Precipitation: zero values percentage",
    "6. Precipitation: extreme values percentage (>100 mm/day)",
    "7. Precipitation: very extreme values percentage (>200 mm/day)",
    "8. Temperature: Tmax >= Tmin consistency percentage",
    "9. Temperature: realistic value ranges",
    "10. Cross-dataset comparisons",
    "11. Visualization generation",
]

for check in checks:
    print(f"  {check}")
    summary["checks_performed"].append(check)

print("\n⚠ KEY FINDINGS:")
print("-" * 40)

# Summarize key findings
findings = []

# Check for negative precipitation across all datasets
for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if dataset_name in quality_stats and "precipitation" in quality_stats[dataset_name]:
        neg_pct = quality_stats[dataset_name]["precipitation"].get("negative_pct", 0)
        neg_count = quality_stats[dataset_name]["precipitation"].get(
            "negative_count", 0
        )
        if neg_count > 0:
            findings.append(
                f"{dataset_name}: {neg_count:,} negative precipitation values ({neg_pct:.4f}%)"
            )
            summary["issues_found"].append(f"{dataset_name} negative precipitation")

# Check temperature consistency violations
for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if (
        dataset_name in quality_stats
        and "temp_consistency" in quality_stats[dataset_name]
    ):
        viol_pct = quality_stats[dataset_name]["temp_consistency"]["violation_pct"]
        viol_count = quality_stats[dataset_name]["temp_consistency"]["violations"]
        if viol_count > 0:
            findings.append(
                f"{dataset_name}: {viol_count:,} Tmax<Tmin violations ({viol_pct:.4f}%)"
            )
            summary["issues_found"].append(
                f"{dataset_name} temperature consistency violations"
            )

if findings:
    for finding in findings:
        print(f"  • {finding}")
else:
    print("  No critical issues found")

print("\n📋 RECOMMENDATIONS:")
print("-" * 40)

recommendations = []

# Add recommendations based on findings
if any(
    "temperature consistency violations" in issue for issue in summary["issues_found"]
):
    recommendations.append(
        "Review temperature data processing - violations found in Tmax/Tmin relationship"
    )

if any("negative precipitation" in issue for issue in summary["issues_found"]):
    recommendations.append(
        "Apply data cleaning to remove/correct negative precipitation values"
    )

# Check for high percentage of extreme values
for dataset_name in ["ERA5", "HIST", "BCSD"]:
    if dataset_name in quality_stats and "precipitation" in quality_stats[dataset_name]:
        extreme_pct = quality_stats[dataset_name]["precipitation"].get("extreme_pct", 0)
        if extreme_pct > 1.0:
            recommendations.append(
                f"{dataset_name}: High percentage of extreme precipitation values - verify if realistic"
            )

if not recommendations:
    recommendations.append("All basic integrity checks passed - proceed with analysis")

for rec in recommendations:
    print(f"  • {rec}")
    summary["recommendations"].append(rec)

# Save comprehensive summary with statistics

with open("comprehensive_integrity_report_with_stats.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n✓ Detailed report saved to 'comprehensive_integrity_report_with_stats.json'")
print("✓ Visualizations saved as PNG files")
print("\n" + "=" * 60)
print("INTEGRITY CHECK COMPLETE")
print("=" * 60)